# Daily Returns Artifact — Construction Spec (shared infrastructure)

**Purpose.** This notebook is the **source-of-truth spec** for the `daily` build, which builds the shared daily panel consumed by every time-series factor build (`beta`, `mom`, `nlb`, `nls`, `resvol`). It is infrastructure, not a factor: it has no descriptor, no standardization, and no USE4 appendix entry of its own — it implements the data conventions the factor specs share (excess returns, ESTU cap-weighted benchmark).

**When to build this.** In the intended order, right after the ESTU and before the time-series style factors — every one of them reads the panel this step produces, so building it up front means each factor just loads a parquet. *Optional detour:* if you would rather feel the machinery before abstracting it, build your first time-series factor (Beta) with the price-loading and return logic inline, then come back and extract it here — that is how the reference build grew, and it trades a little rework for intuition.

**Inputs:** `data/cleaned/prices/*.parquet`, `data/out/estu_monthly.parquet` (from `estu_build.ipynb`), Ken French Data Library (daily RF).

**Deliverables (all `data/out/`):** `daily_returns.parquet`, `crsp_mkt.parquet`, `estu_mkt.parquet`, `ticker_permaticker.parquet`.

## §1. What USE4 requires of this layer — verbatim anchors

### 1a. Excess returns

> *"Excess returns (`r_t − r_ft`) are used"* throughout the USE4 time-series
> estimators (Beta, Momentum, Residual Volatility).

### 1b. Estimation-universe benchmark

> Factor regressions use the **cap-weighted estimation-universe return** —
> the ESTU is the universe over which model parameters are estimated.

---

**That is all the PDF implies for this layer.** Everything else (RF source,
data-quality dedup rules, the FAST/SEQUENTIAL memory strategy) is
❓ **NOT IN PDF** and flagged below.

## §2. End-to-end pipeline

```
┌────────────────────────────────────────────────────────────────────┐
│  UPSTREAM:  estu_build.ipynb  →  estu_monthly.parquet              │
├────────────────────────────────────────────────────────────────────┤
│  STAGE 0:  Setup, parameters                                       │
│  STAGE 1:  Fetch daily RF (Ken French Data Library)                │
│  STAGE 2:  Load prices → excess log returns                        │
│  STAGE 3:  Load ESTU → cap-weighted excess benchmark               │
│  STAGE 4:  Write artifacts + freshness check                       │
└────────────────────────────────────────────────────────────────────┘
```

**Rebuild order:** run after `estu_build.ipynb` and before any time-series factor build.

**PIT discipline:** returns are computed from same-day and prior-day data only; `mcap_lag` is the prior trading day's market cap (used as benchmark weight so the weight is known before the return accrues).

---

**Settled conventions (2026-06-12):**

- ✅ **Total returns verified.** Sharadar `closeadj` is split- **and** dividend-adjusted (stated in the cleaning docs; the DYLD build deliberately uses `closeunadj` for exactly this reason) — `ret` is a total excess return.
- 💡 **Benchmark warm-up snapshot.** The benchmark ownership calendar keeps the 1998-12-31 ESTU snapshot, so January 1999 has a PIT-valid benchmark; factor builds keep their own 1999-01-01 calendars.
- 💡 **Gap-spanning returns kept.** `shift(1)` per ticker spans halts and suspensions: a stock resuming after a 3-week halt contributes one "daily" return covering the gap. Kept deliberately — the information is real, the events are rare, and every consumer's 3σ trim bounds the impact. Revisit if halt-resume returns visibly contaminate DASTD/Beta tails.
- ❗ **RF publication lag.** Ken French RF ships with a lag, so `ret` (and `mkt_ret`) are null for the most recent weeks. Windowed consumers must drop null returns (they do); whole-month consumers skip the affected tail transitions.

## §3. Artifact schemas — what this notebook delivers

**Compression:** zstd, statistics=True. All files written to `data/out/`.

### `daily_returns.parquet` — the primary artifact

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `date` | Date | Trading date |
| `ret` | Float64 | **Excess log return**: `ln(closeadj_t / closeadj_{t-1}) − rf_t` |
| `mcap_lag` | Float64 | Prior trading day's market cap (benchmark weight) |
| `mkt_ret` | Float64 | **ESTU cap-weighted excess benchmark** return on `date` |

### Supporting artifacts

| File | Contents |
|---|---|
| `crsp_mkt.parquet` | `date, mkt_ret` — broad CRSP-proxy benchmark (diagnostics only) |
| `estu_mkt.parquet` | `date, estu_mkt_ret` — the ESTU benchmark series on its own |
| `ticker_permaticker.parquet` | `ticker, permaticker` — symbol map for spot-check lookups |

**Coverage rule:** every (permaticker, date) with a valid prior close is present — the artifact covers the full coverage universe, not just ESTU members.

## §4. STAGE 0 — Setup, imports, paths

Standard imports. Use polars, not pandas, throughout (project convention).

```python
# ───── Parameters ─────────────────────────────────────────────────────────────
CALENDAR_START = date(1999, 1, 1)   # matches all factor builds
```

## §5. STAGE 1 — Daily risk-free rate

**RF** Ken French Data Library, 3-Factor daily CSV, `RF` column
(percent → decimal). Over any 252-day window the RF is approximately constant, so
slope coefficients are insensitive to the choice; it is included for fidelity.

🧪 **VALIDATE:** series spans 1926 → present; mean ≈ 3% annualized; daily values
in [0%, 0.07%].

## §6. STAGE 2 — Prices → excess log returns

**DATA QUALITY**
- Dual-class dedup: keep the highest-mcap row per `(permaticker, date)`
- `ret = ln(closeadj_t / closeadj_{t−1}) − rf_t`; rows with null returns dropped
- `mcap_lag = marketcap.shift(1)` per permaticker

🧪 **VALIDATE:** ~39M rows, ~17,400 tickers, ~6,900 trading days (1998-12 →).

**NOTE** depending on your hardware capabilities, you may want to cosider a multi-threaded approach.

## §7. STAGE 3 — ESTU cap-weighted excess benchmark

✅ **PDF SPEC:** the benchmark is the cap-weighted return of the estimation
universe (see §1b).

💡 **DEFAULT:** for each trading day, weight member excess returns by `mcap_lag`,
membership taken from the **owning signal date** (most recent month-end ≤ date).
The result replaces the broad CRSP-proxy `mkt_ret` in `daily`; the CRSP proxy is
kept as `crsp_mkt.parquet` for diagnostics.

🧪 **VALIDATE:**
- correlation vs broad CRSP proxy > 0.90
- full-sample annualized vol ≈ 15–20%; GFC and COVID windows > 40%
- dates missing the benchmark after the first signal date < 30

## §8. STAGE 4 — Write artifacts

Write the four artifacts (§3) with zstd compression and read-back shape asserts.

🧪 **VALIDATE — freshness contract:** `daily_returns["date"].max()` ≥ the last ESTU signal date; otherwise prices must be refreshed before rebuilding. Downstream factor builds should assert this on load.

## §10. Final summary — what "done" looks like

1. ✅ Four artifacts in `data/out/` with the §3 schemas
2. ✅ Benchmark: CRSP correlation > 0.90, crisis-vol sanity, < 30 missing dates
3. ✅ Freshness: artifact covers the full ESTU signal calendar

Once this passes, the time-series factor builds (`beta`, `mom`, `resvol`, `nlb`, `nls`) can each load `daily_returns.parquet` directly rather than re-deriving the price panel — subject to `estu_build → daily_build → beta → {resvol, nlb}` and `size_build → nls`.